# Chapter 7: Neural Networks

> A single perceptron can only draw a straight line through your data. Stack a hidden layer behind it, and it can draw almost anything.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapter 3 (The Perceptron) &nbsp;|&nbsp; **Time:** ~50 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 8

---

## Learning Objectives

- Explain the biological inspiration behind multi-layer neural networks and the role of the activation (link) function
- Implement forward propagation and back-propagation from scratch for a two-layer network
- Explain why bias terms are necessary, using the XOR problem as a concrete counter-example
- Describe how the number of hidden units and training iterations both act as a form of inductive bias / regularization
- Contrast a two-layer network's representational power with that of the perceptron

## The Problem

The perceptron (Chapter 3) can only represent linear decision boundaries — it fundamentally cannot solve the XOR problem, since no straight line separates XOR's positive and negative points. Decision trees and KNN can express non-linear boundaries, but at the cost of losing the clean, optimization-friendly structure of a linear model.

Neural networks resolve this tension: they chain together perceptron-like units, with a non-linear activation function at each hidden unit, so the overall function becomes non-linear while every individual computation stays simple and differentiable.

## The Concept

```
Inputs x --> Linear: a = Wx + b --> tanh activation --> Linear: y_hat = v.h + c --> Prediction
```

### Key Ideas

- **Two-layer networks are universal approximators**: with enough hidden units, a two-layer network can approximate any continuous function arbitrarily well (Cybenko/Hornik theorem)
- **Back-propagation = gradient descent + the chain rule**: the output layer's gradient is a simple linear-model gradient; the hidden layer's gradient is obtained by pushing the output error backward through the `tanh` derivative
- **Bias terms matter**: `tanh(w·x)` is an *odd* function of `x`. Without a bias, a network can only represent functions that are odd in `x` — and XOR's target is *even* (flipping the sign of both inputs doesn't flip the label), so a bias-free network can never solve it, no matter how many hidden units it has
- **Capacity is controlled by hidden units and training time**: more hidden units, or more gradient steps, let the network fit the training data more closely — this helps only up to a point, after which test performance degrades (overfitting), just like decision tree depth in Chapter 1

## Build It

### Setup

We'll need NumPy for array operations, and several utilities from scikit-learn: the Breast Cancer Wisconsin dataset, a train/test splitter, a feature scaler, scikit-learn's own `MLPClassifier` (as a sanity check against our from-scratch implementation), and an accuracy metric.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

RNG = np.random.RandomState(0)

### Step 1: The Network Class

The architecture is: `D` inputs → `K` hidden units (`tanh`) → 1 output (linear), trained with full-batch gradient descent on squared error.

Weights are initialized with small random values rather than zero, to avoid the trivial symmetric local optimum you'd get from an all-zero start. The hidden bias `b` and output bias `c` are included from the start — as the theory section explains, they are provably required to represent functions like XOR.

In [2]:
class TwoLayerNetFromScratch:
    def __init__(self, n_hidden=10, lr=0.05, n_iter=2000, seed=0):
        self.n_hidden = n_hidden
        self.lr = lr
        self.n_iter = n_iter
        self.seed = seed

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y, dtype=float)
        n, d = X.shape
        rng = np.random.RandomState(self.seed)

        self.W = rng.uniform(-1.0, 1.0, size=(d, self.n_hidden))
        self.b = rng.uniform(-1.0, 1.0, size=self.n_hidden)
        self.v = rng.uniform(-1.0, 1.0, size=self.n_hidden)
        self.c = 0.0

        for _ in range(self.n_iter):
            a = X @ self.W + self.b
            h = np.tanh(a)
            y_hat = h @ self.v + self.c

            e = y - y_hat
            grad_v = -(e[:, None] * h).sum(axis=0)
            grad_c = -e.sum()
            delta = (-e[:, None] * self.v[None, :]) * (1 - h ** 2)
            grad_W = X.T @ delta
            grad_b = delta.sum(axis=0)

            self.v -= self.lr * grad_v / n
            self.c -= self.lr * grad_c / n
            self.W -= self.lr * grad_W / n
            self.b -= self.lr * grad_b / n

        return self

    def decision_function(self, X):
        X = np.asarray(X)
        h = np.tanh(X @ self.W + self.b)
        return h @ self.v + self.c

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1)

### Step 2: Forward Propagation (Algorithm 8.1)

```python
a = X @ self.W + self.b      # pre-activation of hidden units
h = np.tanh(a)               # hidden activations
y_hat = h @ self.v + self.c  # output unit (linear)
```

### Step 3: Back-Propagation (Section 8.2)

```python
e = y - y_hat
grad_v = -(e[:, None] * h).sum(axis=0)                # output-layer gradient
delta = (-e[:, None] * self.v[None, :]) * (1 - h ** 2)  # push error back through tanh'
grad_W = X.T @ delta                                    # hidden-layer gradient
```

Both steps are already implemented inside the class above (`fit`, `decision_function`, and `predict`). The rest of the notebook puts that class to work.

## Use It — Real Data

### Loading and Preparing the Dataset

We'll use the **Breast Cancer Wisconsin (Diagnostic)** dataset, a real, well-known binary classification benchmark bundled with scikit-learn.

Unlike the decision tree in Chapter 1, this network expects continuous inputs, so features are **standardized** (zero mean, unit variance) rather than binarized. Labels are remapped to the book's convention: −1 / +1 instead of 0 / 1.

In [3]:
data = load_breast_cancer()
X, y_raw = data.data, data.target
y = np.where(y_raw == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Total examples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Train/test split: {X_train.shape[0]} / {X_test.shape[0]}")
print(f"Class distribution: {np.sum(y == -1)} negative, {np.sum(y == 1)} positive")

Total examples: 569
Features: 30
Train/test split: 398 / 171
Class distribution: 212 negative, 357 positive


### Sanity Check Against `sklearn.neural_network.MLPClassifier`

To validate the from-scratch implementation, we compare it against scikit-learn's production multi-layer perceptron, using the same number of hidden units and the `tanh` activation.

Note that the two are **not expected to match exactly**: different solvers, initializations, and optimizers can converge to different local optima — a close (not identical) accuracy is the expected outcome.

In [4]:
my_net = TwoLayerNetFromScratch(n_hidden=10, lr=0.1, n_iter=3000, seed=1)
my_net.fit(X_train_s, y_train)
my_pred = my_net.predict(X_test_s)
my_acc = accuracy_score(y_test, my_pred)

sk_mlp = MLPClassifier(
    hidden_layer_sizes=(10,), activation="tanh", solver="lbfgs",
    max_iter=3000, random_state=1,
)
sk_mlp.fit(X_train_s, y_train)
sk_pred = sk_mlp.predict(X_test_s)
sk_acc = accuracy_score(y_test, sk_pred)

print(f"From-scratch net accuracy : {my_acc:.4f}")
print(f"sklearn MLP accuracy      : {sk_acc:.4f}")

From-scratch net accuracy : 0.9649
sklearn MLP accuracy      : 0.9415


### Section 8.1: The Underfitting / Overfitting Tradeoff (vs. Hidden Units)

Just like `max_depth` for decision trees, the number of hidden units `K` is a capacity control knob:

- **Too few hidden units:** the network can't represent the true decision boundary → **underfitting**
- **Too many hidden units:** the network can memorize the training set, including its noise → **overfitting**
- Somewhere in between lies a **sweet spot**, where test accuracy peaks

In [5]:
print(f"{'K':>4} | {'train_acc':>10} | {'test_acc':>9}")
print("-" * 30)

hidden_results = []
for k in [1, 2, 5, 10, 20, 50, 100]:
    net = TwoLayerNetFromScratch(n_hidden=k, lr=0.05, n_iter=3000, seed=1)
    net.fit(X_train_s, y_train)
    train_acc = accuracy_score(y_train, net.predict(X_train_s))
    test_acc = accuracy_score(y_test, net.predict(X_test_s))
    hidden_results.append({'k': k, 'train_acc': train_acc, 'test_acc': test_acc})
    print(f"{k:>4} | {train_acc:>10.4f} | {test_acc:>9.4f}")

   K |  train_acc |  test_acc
------------------------------
   1 |     0.9899 |    0.9649


   2 |     0.9925 |    0.9591
   5 |     0.9899 |    0.9649


  10 |     0.9899 |    0.9649


  20 |     0.9899 |    0.9591


  50 |     0.9975 |    0.9766


 100 |     1.0000 |    0.9591


### Section 8.1: The Underfitting / Overfitting Tradeoff (vs. Training Iterations)

The same tradeoff shows up along a second axis: how long we train. More gradient steps let the network fit the training data more closely, but test accuracy eventually plateaus or degrades once the network starts fitting noise.

In [6]:
print(f"{'iters':>6} | {'train_acc':>10} | {'test_acc':>9}")
print("-" * 32)

iter_results = []
for n_iter in [10, 50, 200, 1000, 3000, 8000]:
    net = TwoLayerNetFromScratch(n_hidden=20, lr=0.1, n_iter=n_iter, seed=1)
    net.fit(X_train_s, y_train)
    train_acc = accuracy_score(y_train, net.predict(X_train_s))
    test_acc = accuracy_score(y_test, net.predict(X_test_s))
    iter_results.append({'n_iter': n_iter, 'train_acc': train_acc, 'test_acc': test_acc})
    print(f"{n_iter:>6} | {train_acc:>10.4f} | {test_acc:>9.4f}")

 iters |  train_acc |  test_acc
--------------------------------
    10 |     0.8492 |    0.8713
    50 |     0.9598 |    0.9532
   200 |     0.9799 |    0.9708
  1000 |     0.9874 |    0.9591


  3000 |     0.9899 |    0.9591


  8000 |     0.9950 |    0.9474


**Reading both tables:** more hidden units and more iterations both increase the model's capacity to fit the training data, but test accuracy eventually plateaus or degrades once the network starts fitting noise in the training set — the same underfitting/overfitting story as Chapter 1, now controlled by two knobs instead of one.

### The XOR Problem: Where the Perceptron Fails and the Network Succeeds

This is the classic proof that two-layer networks beat a single linear unit on non-linear problems. XOR has four points, and no straight line can separate the positive class from the negative class — a linear perceptron is mathematically capped at 75% accuracy here.

In [7]:
X_xor = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]], dtype=float)
y_xor = np.array([-1, 1, 1, -1], dtype=float)

xor_net = TwoLayerNetFromScratch(n_hidden=4, lr=0.5, n_iter=5000, seed=3)
xor_net.fit(X_xor, y_xor)
xor_pred = xor_net.predict(X_xor)

print("Inputs:\n", X_xor)
print("True labels:     ", y_xor)
print("Predicted labels:", xor_pred)
print(f"Accuracy: {accuracy_score(y_xor, xor_pred):.4f}  (a linear perceptron cannot exceed 0.75 here)")

Inputs:
 [[ 1.  1.]
 [ 1. -1.]
 [-1.  1.]
 [-1. -1.]]
True labels:      [-1.  1.  1. -1.]
Predicted labels: [-1  1  1 -1]
Accuracy: 1.0000  (a linear perceptron cannot exceed 0.75 here)


The from-scratch network is competitive with (and here often on par with or ahead of) scikit-learn's `MLPClassifier` on real tumor-diagnosis data, and it solves XOR perfectly once bias terms are included — while a single linear perceptron is mathematically incapable of exceeding 75% accuracy on that same problem.

## When to Use Two-Layer Networks

| API / Function | When to use it |
|---|---|
| `TwoLayerNetFromScratch(n_hidden, lr, n_iter).fit(X, y)` | Small/medium tabular datasets where you want to see exactly what back-prop is doing |
| `sklearn.neural_network.MLPClassifier` | Production use — supports multiple layers, adaptive learning rates, and mini-batching |
| `activation="tanh"` | Classic choice; bounded and zero-centered, unlike ReLU (which this book predates) |
| `hidden_layer_sizes=(K,)` | Choose `K` roughly proportional to `N/D` (Section 8.1 heuristic) as a starting point, then tune on held-out data |

## Exercises

1. Replace the squared-error loss with logistic loss (Chapter 6) and re-derive the back-propagation gradients — how do `grad_v` and `delta` change?
2. Add a second hidden layer to `TwoLayerNetFromScratch` and re-run the XOR experiment — does it still need bias terms at every layer?
3. Reproduce Figure 3.3-style overfitting curves by plotting train/test accuracy against `n_iter` at several fixed values of `n_hidden`.

## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **Hidden Unit** | "Just an extra output" | An intermediate neuron whose activation is a non-linear function of a linear combination of inputs, never directly observed as a label |
| **Back-propagation** | "A special neural-network trick" | Ordinary gradient descent combined with the chain rule, applied to a computation graph with more than one layer |
| **Bias Term** | "A minor implementation detail" | A learnable constant offset that lets a unit's decision threshold move away from the origin — provably required to represent even simple functions like XOR |
| **Universal Approximation** | "Any network can learn anything, easily" | A theorem about *representational* capacity (a function exists), which says nothing about whether gradient descent will actually *find* that function |

## Summary

- Neural networks chain together perceptron-like linear units with a non-linear activation (`tanh`), turning a family of simple, differentiable computations into a universal function approximator
- **Back-propagation** is just gradient descent plus the chain rule, applied one layer at a time
- **Bias terms** aren't a minor detail — without them, a network can only represent odd functions of its input, and provably cannot solve XOR
- **Hidden units** and **training iterations** are two knobs controlling the same underfitting/overfitting tradeoff seen with decision tree depth in Chapter 1
- Our from-scratch implementation is competitive with scikit-learn's `MLPClassifier` on real tumor-diagnosis data, and solves XOR perfectly — something no linear perceptron can do

---

**Next:** Chapter 8 — beyond two layers